# 📱 Google Play Store — ML Analysis Notebook
### Mobile App Success Predictor & Recommender
**Tools:** Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors

plt.rcParams["figure.facecolor"] = "#0e1117"
plt.rcParams["axes.facecolor"]   = "#161b22"
plt.rcParams["text.color"]        = "#e0e0e0"
plt.rcParams["axes.labelcolor"]   = "#8b949e"
plt.rcParams["xtick.color"]       = "#8b949e"
plt.rcParams["ytick.color"]       = "#8b949e"
plt.rcParams["axes.edgecolor"]    = "#30363d"
print("Imports OK")


## 1. Data Loading & Inspection

In [ ]:
df_raw = pd.read_csv("googleplaystore.csv")
print("Shape:", df_raw.shape)
df_raw.head()


In [ ]:
# Null values
df_raw.isnull().sum()


## 2. Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# ── Duplicates ──────────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(subset="App", keep="last", inplace=True)
print(f"Dropped {before - len(df):,} duplicate rows")

# ── Drop missing target ──────────────────────────────────────────────────
df.dropna(subset=["Rating"], inplace=True)
df = df[pd.to_numeric(df["Rating"], errors="coerce").between(1, 5)]
df["Rating"] = df["Rating"].astype(float)

# ── Size → MB ───────────────────────────────────────────────────────────
def parse_size(s):
    s = str(s).strip()
    if s in ("Varies with device", "nan", ""):
        return np.nan
    if s.endswith("M"): return float(s[:-1])
    if s.endswith("k"): return float(s[:-1]) / 1024
    return np.nan

df["Size_MB"] = df["Size"].apply(parse_size)
df["Size_MB"] = df["Size_MB"].fillna(df["Size_MB"].median())
print("Size_MB NaN remaining:", df["Size_MB"].isna().sum())

# ── Installs → int ──────────────────────────────────────────────────────
df["Installs_Clean"] = (
    df["Installs"]
    .str.replace(r"[+,]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0).astype(int)
)

# ── Price → float ───────────────────────────────────────────────────────
df["Price_Clean"] = (
    df["Price"].astype(str)
    .str.replace("$", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0.0)
)

# ── Reviews → int ───────────────────────────────────────────────────────
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce").fillna(0).astype(int)

# ── Drop rare nulls ──────────────────────────────────────────────────────
df.dropna(subset=["Content Rating", "Type"], inplace=True)

# ── Genre (first tag) ────────────────────────────────────────────────────
df["Genre_Clean"] = df["Genres"].str.split(";").str[0].str.strip()

df.reset_index(drop=True, inplace=True)
print("Final shape:", df.shape)
df[["App","Category","Rating","Size_MB","Installs_Clean","Price_Clean"]].head()


## 3. Exploratory Data Analysis

In [ ]:
# ── Reviews vs Ratings scatter ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
sample = df[df["Reviews"] < df["Reviews"].quantile(0.98)].sample(2000, random_state=42)
ax.scatter(np.log1p(sample["Reviews"]), sample["Rating"],
           alpha=0.3, s=14, c="#7c3aed")
ax.set_xlabel("log(Reviews + 1)")
ax.set_ylabel("Rating")
ax.set_title("Reviews vs Ratings (log scale)", color="#c4b5fd")
plt.tight_layout(); plt.show()
corr = sample[["Reviews","Rating"]].corr().iloc[0,1]
print(f"Pearson correlation (Reviews vs Rating): {corr:.4f}")


In [ ]:
# ── Rating distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["Rating"], bins=30, color="#4f46e5", edgecolor="#0e1117", alpha=0.85)
ax.axvline(df["Rating"].mean(), color="#10b981", linestyle="--",
           label=f"Mean: {df['Rating'].mean():.2f}")
ax.set_xlabel("Rating"); ax.set_ylabel("Count")
ax.set_title("Rating Distribution", color="#c4b5fd")
ax.legend(facecolor="#1c2333", edgecolor="#30363d")
plt.tight_layout(); plt.show()


In [ ]:
# ── Category vs Avg Rating (top 15) ─────────────────────────────────
top_cats = df.groupby("Category")["Rating"].mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_cats.index[::-1], top_cats.values[::-1],
        color=plt.cm.plasma(np.linspace(0.2,0.85,15)))
ax.set_xlabel("Avg Rating")
ax.set_title("Top 15 Categories by Average Rating", color="#c4b5fd")
plt.tight_layout(); plt.show()


## 4. Feature Engineering & Encoding

In [ ]:
le_cat   = LabelEncoder()
le_cr    = LabelEncoder()
le_type  = LabelEncoder()
le_genre = LabelEncoder()

df["Category_Enc"]      = le_cat.fit_transform(df["Category"])
df["ContentRating_Enc"] = le_cr.fit_transform(df["Content Rating"])
df["Type_Enc"]          = le_type.fit_transform(df["Type"])
df["Genre_Enc"]         = le_genre.fit_transform(df["Genre_Clean"])

feature_cols = ["Category_Enc", "Size_MB", "Installs_Clean",
                "Price_Clean", "ContentRating_Enc", "Type_Enc",
                "Genre_Enc", "Reviews"]

X = df[feature_cols].values
y = df["Rating"].values
print("X shape:", X.shape, "  y shape:", y.shape)


## 5. Supervised Learning — Random Forest Regressor

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error  (MAE) : {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


In [ ]:
# ── Actual vs Predicted ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred, alpha=0.3, s=12, c="#7c3aed")
ax.plot([1,5],[1,5], "--", color="#10b981", linewidth=1.5, label="Perfect fit")
ax.set_xlabel("Actual Rating"); ax.set_ylabel("Predicted Rating")
ax.set_title("Actual vs Predicted Ratings", color="#c4b5fd")
ax.legend(facecolor="#1c2333", edgecolor="#30363d")
plt.tight_layout(); plt.show()


In [ ]:
# ── Feature Importances ──────────────────────────────────────────────
fi_df = pd.DataFrame({
    "Feature": ["Category","Size(MB)","Installs","Price",
                "Content Rating","Type","Genre","Reviews"],
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fi_df["Feature"], fi_df["Importance"],
        color=["#7c3aed" if v > fi_df["Importance"].median() else "#4f46e5"
               for v in fi_df["Importance"]])
ax.set_xlabel("Importance Score")
ax.set_title("Random Forest Feature Importances", color="#c4b5fd")
plt.tight_layout(); plt.show()
fi_df.sort_values("Importance", ascending=False)


## 6. Unsupervised Learning — KNN App Recommender

In [ ]:
# ── Build KNN model ──────────────────────────────────────────────────
rec_features = ["Category_Enc","Genre_Enc","Price_Clean",
                "ContentRating_Enc","Size_MB"]

scaler = StandardScaler()
X_rec  = scaler.fit_transform(df[rec_features].values)

knn = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute")
knn.fit(X_rec)
print("KNN model trained on", X_rec.shape[0], "apps")


In [ ]:
def recommend_apps(app_name, n=5):
    """Return top-n most similar apps using KNN cosine similarity."""
    matches = df[df["App"].str.lower() == app_name.lower()]
    if matches.empty:
        matches = df[df["App"].str.lower().str.contains(app_name.lower(), na=False)]
    if matches.empty:
        return f"App '{app_name}' not found."

    idx = matches.index[0]
    query = scaler.transform(df.loc[[idx], rec_features].values)
    distances, indices = knn.kneighbors(query, n_neighbors=n+1)
    recs = df.iloc[indices[0][1:]].copy()
    recs["Similarity"] = (1 - distances[0][1:]).round(3)
    return recs[["App","Category","Genre_Clean","Rating",
                 "Price_Clean","Similarity"]].rename(
        columns={"Genre_Clean":"Genre","Price_Clean":"Price"})

# ── Test the recommender ─────────────────────────────────────────────
recommend_apps("Clash of Clans")


In [ ]:
recommend_apps("Instagram")


## 7. Predict Rating for a New App

In [ ]:
def predict_new_app(category, size_mb, price, content_rating, app_type, genre):
    def safe_enc(le, val):
        return le.transform([val])[0] if val in le.classes_ else 0
    X_new = np.array([[
        safe_enc(le_cat, category),
        size_mb, 1000,
        price,
        safe_enc(le_cr, content_rating),
        safe_enc(le_type, app_type),
        safe_enc(le_genre, genre),
        100
    ]])
    return round(float(rf.predict(X_new)[0]), 2)

# Example: Predict rating for a new free game
pred = predict_new_app("GAME", 25.0, 0.0, "Everyone", "Free", "Action")
print(f"Predicted Rating: {pred:.2f} / 5.0")


## 8. Summary

| Metric | Value |
|--------|-------|
| Random Forest MAE | See above |
| Random Forest RMSE | See above |
| Top Feature | Reviews / Installs |
| Dataset size (cleaned) | ~8,200 apps |

**Key findings:**
- **Reviews** and **Installs** are the strongest predictors of rating.
- **Free apps** slightly outperform paid in avg rating.
- **Events, Education, Art & Design** categories have the highest avg ratings.
- KNN cosine similarity retrieves semantically similar apps with high accuracy.
